<h1>Table des matières<span class="tocSkip"></span></h1>
<div class="toc" style="margin-top: 1em;"><ul class="toc-item"><li><span><a href="#Sujet-de-TP-:-modèle-de-langage-avec-un-RNN-&quot;Vanilla&quot;" data-toc-modified-id="Sujet-de-TP-:-modèle-de-langage-avec-un-RNN-&quot;Vanilla&quot;-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Sujet de TP : modèle de langage avec un RNN "Vanilla"</a></span></li><li><span><a href="#Préparation-des-données" data-toc-modified-id="Préparation-des-données-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Préparation des données</a></span></li><li><span><a href="#imports-et-accès-à-un-GPU" data-toc-modified-id="imports-et-accès-à-un-GPU-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>imports et accès à un GPU</a></span></li><li><span><a href="#Chargement-du-corpus" data-toc-modified-id="Chargement-du-corpus-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Chargement du corpus</a></span></li><li><span><a href="#Créer-la-classe-du-modèle" data-toc-modified-id="Créer-la-classe-du-modèle-5"><span class="toc-item-num">5&nbsp;&nbsp;</span>Créer la classe du modèle</a></span></li><li><span><a href="#Instancier-le-modèle" data-toc-modified-id="Instancier-le-modèle-6"><span class="toc-item-num">6&nbsp;&nbsp;</span>Instancier le modèle</a></span></li><li><span><a href="#Apprentissage" data-toc-modified-id="Apprentissage-7"><span class="toc-item-num">7&nbsp;&nbsp;</span>Apprentissage</a></span></li><li><span><a href="#Tester-le-modèle-sur-des-phrases" data-toc-modified-id="Tester-le-modèle-sur-des-phrases-8"><span class="toc-item-num">8&nbsp;&nbsp;</span>Tester le modèle sur des phrases</a></span></li></ul></div>

# Sujet de TP : modèle de langage avec un RNN "Vanilla"

Dans ce TP, nous allons créer un RNN simple (un "Vanilla RNN") qui prédit le mot qui suit un début de phrase. Il s'agit d'un modèle de langage.

Nous allons entraîner ce RNN sur un tout petit sous-ensemble de textes provenant du corpus Librivox French (https://librivox.org) qui regroupe des audiobooks.

Une fois entraîné, vous pouvez compléter un début de phrase en faisant des prédictions avec le modèle.

Nous utilisons les balises < unk > et < eos > pour les mots qui ne sont pas dans notre vocabulaire et pour indiquer une fin de phrase, respectivement.
    
Nous avons restreint le vocabulaire à la liste de mots apparaissant au moins 4 fois dans Librivox.

# imports et accès à un GPU

In [2]:
import os
import torch
import torch.nn.functional as F
import torch.nn as nn

import math
import time
import re

Avec ou sans GPU ?

Il est recommandé d'exécuter ce code sur GPU :<br>
* Temps pour 1 epoch sur CPU : 153 sec ( 2.55 min)<br>
* Temps pour 1 epoch sur GPU : 8.4 sec avec une GeForce GTX 1080 Ti <br>

In [3]:
# os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
# os.environ["CUDA_VISIBLE_DEVICES"]="0"

In [4]:
device= torch.device("cuda")
# device= torch.device("cpu")
print(device) #, torch.cuda.get_device_name(device=None))

cuda


In [5]:
torch.cuda.empty_cache()

# Préparation des données

Nous vous fournissons les données précalculées ci-dessous: chaque mot est associé à un identifiant unique, et les phrases sont alors stockées comme des vecteurs d'entiers

# Chargement du corpus


Téléchargement des données

In [6]:
!wget -O dataset.zip https://www.irit.fr/~Thomas.Pellegrini/ens/RNN/librivox_fr_PTfiles.zip
!ls -alth dataset.zip
!unzip -qq dataset.zip

--2026-04-10 06:31:27--  https://www.irit.fr/~Thomas.Pellegrini/ens/RNN/librivox_fr_PTfiles.zip
Resolving www.irit.fr (www.irit.fr)... 141.115.28.2
Connecting to www.irit.fr (www.irit.fr)|141.115.28.2|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 947256 (925K) [application/zip]
Saving to: ‘dataset.zip’

dataset.zip         100%[===================>] 925.05K  5.38MB/s    in 0.2s    

2026-04-10 06:31:28 (5.38 MB/s) - ‘dataset.zip’ saved [947256/947256]

-rw-r--r-- 1 root root 926K Apr  8  2024 dataset.zip


Vérifier que vous avez bien généré les données :

In [7]:
path_data='./'

flag_train_data = os.path.isfile(path_data + 'librivox_fr/train_data.pt')
flag_test_data = os.path.isfile(path_data + 'librivox_fr/test_data.pt')

flag_idx2word = os.path.isfile(path_data + 'librivox_fr/idx2word.pt')
flag_word2idx = os.path.isfile(path_data + 'librivox_fr/word2idx.pt')

if flag_idx2word==False or flag_test_data==False or flag_train_data==False or flag_word2idx==False:
    print('Librivox_fr dataset manquant')


Charger train_data et test_data et afficher les dimensions des deux tensors

In [8]:
train_data  =  torch.load(path_data+'librivox_fr/train_data.pt')
test_data   =  torch.load(path_data+'librivox_fr/test_data.pt')

# TODO : afficher les dimensions des deux tensors
print(train_data.shape)
print(test_data.shape)

torch.Size([20542, 20])
torch.Size([1496, 1])


Charger les dictionnaires idx2word et word2idx :

In [9]:
word2idx  =  torch.load(path_data + 'librivox_fr/word2idx.pt')
idx2word  =  torch.load(path_data + 'librivox_fr/idx2word.pt')

La première phrase du texte train_librivox_fr_50words_max_15200.txt est :

"enfant si j'étais roi je donnerais l'empire et mon char et mon sceptre et mon peuple à genoux et ma couronne d'or et mes bains de porphyre et mes flottes à qui la mer ne peut suffire"

**Comment est-elle stockée dans le tenseur train_data ?**

 Afficher les 50 premiers mots de train_data, 10 mots par ligne. Précéder chaque mot par son identifiant entier dans le dictionnaire. Vous devez obtenir un affichage similaire à :

 ```1:enfant 2:si 3:j'étais 4:roi 5:je 6:donnerais 7:l'empire 8:et 9:mon 10:char```

 ```8:et 9:mon 11:sceptre 8:et 9:mon 12:peuple 13:à 14:genoux 8:et 15:ma```

 ...

In [22]:
count_words = 0
for row_idx in range(train_data.shape[0]):
    for col_idx in range(train_data.shape[1]):
        if count_words >= 50:
            break
        word_id = train_data[row_idx, col_idx].item()
        word = idx2word[word_id]
        print(f"{word_id}:{word}", end=" ")
        count_words += 1
        if count_words % 10 == 0:
            print()
    if count_words >= 50:
        break

1:enfant 28:<eos> 170:encore 640:rue 446:oiseau 210:dans 138:jour 1734:part 365:temps 20:de 
20:de 2885:certaines 2678:minute 193:peu 773:vieille 25:ne 377:mais 328:est 3061:parlant 4869:îles 
2:si 457:nous 43:le 13:à 122:sur 256:son 5:je 4486:zobéide 36:avec 23:la 
34:terre 0:<unk> 124:sans 1956:l_heure 235:joie 4466:dormez 0:<unk> 7351:dévoré 369:toujours 0:<unk> 
3:j’étais 25:ne 1105:salut 1304:gauche 683:l_arbre 3122:carré 2883:crois 13:à 23:la 4091:jeunesse 


**Comment est indiquée la fin de cette phrase ?**

**Comment sont gérés les mots qui ne sont pas dans notre dictionnaire (les mots apparaissant moins de quatre fois) ?**

La fin de chaque phrase est indiquée par le token `<eos>` (end of sentence)

Les mots qui ne sont pas dans notre dictionnaire (ceux qui apparaissent moins de quatre fois dans le corpus d'entraînement) sont remplacés par le token `<unk>` (unknown)

**Où est stockée la deuxième phrase, qui commence par "pour un regard de vous" ?**

**En déduire comment les données sont présentées au futur modèle.**


In [18]:
max(torch.unique(train_data)), len((torch.unique(train_data)))

(tensor(9573), 9574)

In [19]:
max(torch.unique(test_data)), len((torch.unique(test_data)))

(tensor(9573), 506)

**Notez qu'il y a un problème avec les données de test : lequel ? Comment pourrait-on le réparer ?**

Quelques constantes associées au dataset

In [20]:
bs = 20 # taille des batches : fixée ici dans le prétraitement
seq_length = 15 # taille des "phrases" à donner au réseau

vocab_size = 9574

# Créer la classe du modèle

1.   Compléter la définition du modèle à trois couches suivant, en indiquant les bonnes dimensions.


2.   Compléter la définition de la fonction forward.

In [24]:
class three_layer_recurrent_net(nn.Module):

    def __init__(self, embed_dim, hidden_dim, vocab_size):

        super(three_layer_recurrent_net, self).__init__()

        self.layer1 = nn.Embedding(vocab_size, embed_dim)
        self.layer2 = nn.GRU(embed_dim , hidden_dim, num_layers = 1)
        self.layer3 = nn.Linear(hidden_dim, vocab_size)

    def forward(self, word_seq, h_init ):

        g_seq               = self.layer1(word_seq)
        h_seq , h_final     = self.layer2(g_seq, h_init)
        score_seq           = self.layer3(h_seq)

        return score_seq,  h_final

# Instancier le modèle

Il est possible d'accéder aux paramètres d'un modèle à l'aide de la méthode net.parameters().


Compléter la fonction display_num_param qui affiche le nombre de paramètres d'un réseau donné en argument (net).

In [25]:
def display_num_param(net):
    num_params = 0
    for param in net.parameters():
        num_params += param.numel()
    print(f"Total number of parameters: {num_params}")

Instancier le réseau dans une variable appelée net. Le réseau doit avoir une taille d'embedding à 100, et 150 neurones pour la couche récurrente. Combien de paramètres au total contient le modèle?

In [26]:
# TODO
embed_dim = 100
hidden_dim = 150
net = three_layer_recurrent_net(embed_dim, hidden_dim, vocab_size)
display_num_param(net)

Total number of parameters: 2516474


Envoyer le modèle sur le GPU (si vous utilisez un gpu)

In [27]:
net = net.to(device)

Initialiser les poids de la couche embedding et de la couche linéaire avec une distribution uniforme sur [-0.1, 0.1]

In [28]:
net.layer1.weight.data.uniform_(-0.1, 0.1)

net.layer3.weight.data.uniform_(-0.1, 0.1)

print('')

# Apprentissage

Définir la fonction de coût entropie croisée et les hyperparamètres suivants :
* learning rate initial : my_lr=1
* taille des séquences : seq_length=35

In [29]:
criterion = nn.CrossEntropyLoss()

my_lr = 1

seq_length = 35

Lors de l'apprentissage, pour éviter le phénomène d'explosion du gradient, nous allons utiliser une fonction qui normalise les valeurs du gradient

In [30]:
def normalize_gradient(net):

    grad_norm_sq=0

    for p in net.parameters():
        grad_norm_sq += p.grad.data.norm()**2

    grad_norm=math.sqrt(grad_norm_sq)

    if grad_norm<1e-4:
        net.zero_grad()
        print('norme du gradient proche de zéro')
    else:
        for p in net.parameters():
             p.grad.data.div_(grad_norm)

    return grad_norm

Voici une fonction qui évaluerait le réseau sur le jeu de test (non-utilisée à cause du problème dans les données de test)

In [31]:
def eval_on_test_set():

    test_bs = 1
    running_loss=0
    num_batches=0

    h = torch.zeros(1, 1, hidden_dim)

    h=h.to(device)

    for count in range( 0 , 1496-seq_length ,  seq_length) :

        minibatch_data =  test_data[ count   : count+seq_length   ]
        minibatch_label = test_data[ count+1 : count+seq_length+1 ]

        minibatch_data=minibatch_data.to(device)
        minibatch_label=minibatch_label.to(device)

        scores, h  = net( minibatch_data, h )

        minibatch_label =   minibatch_label.view(  test_bs*seq_length )
        scores          =            scores.view(  test_bs*seq_length , vocab_size)

        loss = criterion(  scores ,  minibatch_label )

        h = h.detach()

        running_loss += loss.item()
        num_batches += 1

    total_loss = running_loss/num_batches
    print('test: exp(loss) = %.1f'%math.exp(total_loss)  )

Compléter la boucle d'apprentissage aux endroits indiqués par ?? et entraîner le modèle sur 10 epochs

In [32]:
start=time.time()

for epoch in range(10):

    # garder le learning rate à 1.0 pour les 4 premières epochs, puis diviser par 1.1 à chaque epoch
    if epoch >= 4:
        my_lr = 1.0/(1.1**epoch)

    # créer un nouvel optimizer de type SGD et lui passer les paramètres du modèle et le learning rate actualisé.
    optimizer=torch.optim.SGD(net.parameters(), my_lr )

    # initialisation du coût et du nombre de batchs à chaque nouvelle epoch
    running_loss=0
    num_batches=0

    # initialiser h par un vecteur de zéros avec les bonnes dimensions requises.
    # Voir la doc https://pytorch.org/docs/stable/generated/torch.nn.GRU.html
    h = torch.zeros(1, bs, hidden_dim)

    # envoi au gpu
    h=h.to(device)

    for count in range( 0 , 20542-seq_length ,  seq_length):

        # Mettre les gradients à zéro
        optimizer.zero_grad()

        # créer un minibatch
        minibatch_data =  train_data[ count   : count+seq_length   ]
        minibatch_label = train_data[ count+1 : count+seq_length+1 ]

        # envoi au gpu
        minibatch_data=minibatch_data.to(device)
        minibatch_label=minibatch_label.to(device)

        # Detacher h pour ne pas backpropager sur toutes les séquences depuis le début de l'epoch
        h=h.detach()
        # Dire à Pytorch de tracker les opérations sur h pour le minibatch courant
        h=h.requires_grad_()

        # Réaliser une Passe forward
        scores, h  = net(minibatch_data, h)

        # Reshape les tenseurs scores et labels pour obtenir une longueur de bs*seq_length
        scores          =            scores.view(  bs*seq_length , vocab_size)
        minibatch_label =   minibatch_label.view(  bs*seq_length )

        # Calculer la loss moyenne
        loss = criterion(scores, minibatch_label)

        # Passe backward pour calculer les gradients dL/dR, dL/dV et dL/dW
        loss.backward()

        # Normaliser les gradients et faire une itération de SGD : R=R-lr(dL/dR), V=V-lr(dL/dV), ...
        normalize_gradient(net)
        optimizer.step()

        # Actualiser le coût par epoch et le nb de batches traités
        running_loss += loss.item()
        num_batches += 1

    # calcul du coût sur tout le training set
    total_loss = running_loss / num_batches
    elapsed = time.time() - start

    print('')
    print('epoch=',epoch, '\t time=%.2f s'%elapsed,'\t lr=%.6f'%my_lr, '\t exp(loss)=%.1f'%math.exp(total_loss))

    # Estimer la performance sur le jeu de test
    eval_on_test_set()


epoch= 0 	 time=8.56 s 	 lr=1.000000 	 exp(loss)=673.7
test: exp(loss) = 421.3

epoch= 1 	 time=11.05 s 	 lr=1.000000 	 exp(loss)=346.5
test: exp(loss) = 278.4

epoch= 2 	 time=14.08 s 	 lr=1.000000 	 exp(loss)=250.5
test: exp(loss) = 244.2

epoch= 3 	 time=16.70 s 	 lr=1.000000 	 exp(loss)=205.4
test: exp(loss) = 213.2

epoch= 4 	 time=19.20 s 	 lr=0.683013 	 exp(loss)=174.3
test: exp(loss) = 196.3

epoch= 5 	 time=21.72 s 	 lr=0.620921 	 exp(loss)=158.6
test: exp(loss) = 188.6

epoch= 6 	 time=24.23 s 	 lr=0.564474 	 exp(loss)=147.0
test: exp(loss) = 183.1

epoch= 7 	 time=26.78 s 	 lr=0.513158 	 exp(loss)=137.9
test: exp(loss) = 179.2

epoch= 8 	 time=29.37 s 	 lr=0.466507 	 exp(loss)=130.5
test: exp(loss) = 176.5

epoch= 9 	 time=31.89 s 	 lr=0.424098 	 exp(loss)=124.5
test: exp(loss) = 174.5


**Question** : quelle est la taille des tenseurs suivants ?

* minibatch_data, minibatch_label,
* h et scores avant le reshape
* h et scores après le reshape


# Tester le modèle sur des phrases

Voici une fonction qui prend une phrase et qui la convertit en tenseur exploitable pour le réseau

In [ ]:
def sentence2vector_librivox_fr(sentence):
    words = sentence.split()
    x = torch.LongTensor(len(words),1)
    for idx, word in enumerate(words):
        word = re.sub("'", "_", word)
        if word not in word2idx:
            print('Vous avez entrer un mot hors-vocabulaire.')
            print('--> Enlever lettres majuscules et ponctuation')
            print("mot --> <unk> avec index 0")
            x[idx,0]=0
        else:
            x[idx,0]=word2idx[word]
    return x


In [ ]:
sentence1 = "on entendait vaguement au dehors les"

sentence2 = "hier je luttai de la sorte contre le grand"

sentence3 = "il connaissait la route et nous avons"

# ou bien créer votre propre phrase. Il ne fauit pas utiliser de majuscules ni de ponctuation.
# Chaque mot doit être dans le lexique.
sentence4= "la vie est"

# Choisir le phrase ici :
mysentence = sentence1

Convertir la phrase choisie et envoi au GPU

In [ ]:
minibatch_data=sentence2vector_librivox_fr(mysentence)

minibatch_data=minibatch_data.to(device)

print(minibatch_data)

tensor([[ 366],
        [1383],
        [1884],
        [  72],
        [ 127],
        [  37]], device='cuda:0')


Définir un hidden state initial à zero, et exécuter le RNN sur la phrase

In [ ]:
h = torch.zeros(1, 1, hidden_dim)
h=h.to(device)

scores , h = net( minibatch_data , h )

Écrire une fonction show_next_word qui prend en entrée scores et qui affiche les 30 mots les plus probables prédits par le réseau, en indiquant leur probabilité par un pourcentage.

L'affichage doit ressembler à :

```
on entendait vaguement au dehors les ...

6.0%	 autres
3.2%	 <unk>
2.0%	 hommes
1.9%	 plus
1.6%	 yeux
...
```

Vous utiliserez la fonction torch.topk()
Aide : https://pytorch.org/docs/stable/torch.html?highlight=topk#torch.topk


In [ ]:
def show_next_word(scores):
    # scores shape: [seq_length, 1, vocab_size]
    # We want the prediction for the last word, which is scores[-1].
    # scores[-1] has shape [1, vocab_size]
    last_word_scores = scores[-1]

    # Apply softmax to get probabilities
    # prob_last_word will have shape [1, vocab_size]
    prob_last_word = F.softmax(last_word_scores, dim=1) # dim=1 for the vocab_size dimension

    # Get the top 30 probabilities and their indices
    # top_probs will have shape [1, 30], top_indices will have shape [1, 30]
    top_probs, top_indices = torch.topk(prob_last_word, 30)

    # Squeeze to remove the batch dimension (which is 1)
    # top_probs_squeezed will have shape [30], top_indices_squeezed will have shape [30]
    top_probs_squeezed = top_probs.squeeze()
    top_indices_squeezed = top_indices.squeeze()

    for i in range(30):
        # Now top_probs_squeezed[i] and top_indices_squeezed[i] are scalar tensors,
        # so .item() can be called.
        percentage = top_probs_squeezed[i].item() * 100
        word = idx2word[top_indices_squeezed[i].item()]
        print("{:.1f}%\t{}".format(percentage, word))


Afficher la prédiction du prochain mot par le réseau

In [ ]:
print(mysentence, '... \n')

show_next_word(scores)

on entendait vaguement au dehors les ... 

4.1%	autres
3.5%	<unk>
3.5%	deux
1.6%	yeux
1.6%	hommes
1.3%	trois
1.3%	plus
1.2%	uns
1.0%	enfants
0.9%	maoris
0.9%	mains
0.8%	flots
0.7%	captifs
0.7%	quatre
0.7%	indigènes
0.7%	petits
0.7%	mêmes
0.6%	pieds
0.6%	eaux
0.5%	dames
0.5%	arbres
0.5%	sauvages
0.5%	jeunes
0.4%	murs
0.4%	princes
0.4%	épaules
0.4%	fugitifs
0.4%	pauvres
0.4%	convives
0.4%	nuages


Les mots < unk > et < eos > seront presque toujours les plus probables.

Voici une fonction get_next_word, variante de show_next_word qui retourne le mot le plus fréquent.

Si ce mot est < unk > ou < eos >, cette fonction retourne le mot suivant le plus probable, à moins que ce soit < eos > avec une proba plus grande que 50 %, auquel cas on termine la phrase.

In [ ]:
def get_next_word(scores):

    prob=F.softmax(scores,dim=2)
    num_word_display = 3
    p=prob[-1].squeeze()
    p, word_idx = torch.topk(p, num_word_display)

    for i,idx in enumerate(word_idx):
        percentage = p[i].item()*100
        word = idx2word[idx.item()]
        print(  "{:.1f}%\t".format(percentage),  word )
        if word == '<eos>' and percentage > 50.:
          return word
        if word != '<eos>' and word != '<unk>':
          return word

In [1]:
next_word = get_next_word(scores)

NameError: name 'get_next_word' is not defined

Écrire un bout de code qui prédit une phrase entière à partir de mysentence.

Cette phrase sera considérée comme terminée sur le modèle prédit < eos > avec une proba plus grande que 50%, ou bien après 10 itérations maximum.

In [ ]:
mysentence = "on entendait vaguement au dehors les"

# mysentence = "hier je luttai de la sorte contre le grand"

# mysentence = "il connaissait la route et nous avons"

# mysentence= "la vie est très"

print(mysentence + '...')

i= 0
not_finished = True

h = torch.zeros(1, 1, hidden_dim)
h=h.to(device)

while i < 10 and not_finished :
    ## TODO
